# Edge Discovery via Claude API

Uses the Claude API to propose new relationships between concepts in the calculus knowledge graph.

**How it works:**
1. Load the existing graph (concepts + edges)
2. For each pair of concepts in the same or adjacent chapters that DON'T already have an edge, ask Claude to classify the relationship
3. Claude returns: prerequisite, uses, related_to, or none — plus a reason
4. You review the proposals and accept/reject them

**Author:** Catherine Cho — Hanlon/Coulter Scholars

In [ ]:
%pip install networkx anthropic

In [ ]:
import json
import time
import os
from anthropic import Anthropic

# ========================================
# SET YOUR API KEY HERE
# Get one at: https://console.anthropic.com/
# ========================================
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE"))

print("API client ready.")

## Load the existing graph

Run the calculus_knowledge_graph.ipynb notebook first to generate `calculus_kg.json`.
Then update the path below if needed.

In [ ]:
import networkx as nx

# Load the graph you built in the other notebook
with open("calculus_kg.json", "r") as f:
    data = json.load(f)

G = nx.node_link_graph(data, directed=True, edges="edges")
print(f"Loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Get only concept nodes (not chapters or topics)
concepts = {n: d for n, d in G.nodes(data=True) if d.get("type") == "concept"}
print(f"Concepts: {len(concepts)}")

## Build the concept-pair batches

We don't want to check ALL pairs — that's O(n²) ≈ 10,000 pairs for 100 concepts.
Instead, we only check pairs that:
1. Don't already have a direct edge
2. Are in the same chapter or adjacent chapters (concepts 4 chapters apart are unlikely to have a direct relationship)

We also batch 10 pairs per API call to reduce costs.

In [ ]:
# Map each concept to its chapter number
def get_chapter_num(concept_name):
    """Follow is_part_of edges: concept → topic → chapter, extract chapter number."""
    for succ in G.successors(concept_name):
        edge = G.edges[concept_name, succ]
        if edge.get("edge_type") == "is_part_of":
            topic = succ
            # Topic name starts with section number like "3.6"
            try:
                return int(topic.split(".")[0].strip())
            except ValueError:
                # It's a chapter node like "Ch3: Derivatives"
                for succ2 in G.successors(topic):
                    edge2 = G.edges[topic, succ2]
                    if edge2.get("edge_type") == "is_part_of":
                        try:
                            return int(succ2.split(":")[0].replace("Ch", "").strip())
                        except:
                            pass
    return None

concept_chapters = {}
for c in concepts:
    ch = get_chapter_num(c)
    if ch is not None:
        concept_chapters[c] = ch

print(f"Mapped {len(concept_chapters)} concepts to chapters")

# Show distribution
from collections import Counter
ch_counts = Counter(concept_chapters.values())
for ch in sorted(ch_counts):
    print(f"  Chapter {ch}: {ch_counts[ch]} concepts")

In [ ]:
# Find candidate pairs: same chapter or adjacent chapters, no existing edge
existing_edges = set(G.edges())
candidate_pairs = []

concept_list = sorted(concept_chapters.keys())
for i, c1 in enumerate(concept_list):
    for c2 in concept_list[i+1:]:
        ch1, ch2 = concept_chapters[c1], concept_chapters[c2]
        # Same chapter or adjacent chapter
        if abs(ch1 - ch2) <= 1:
            # No existing edge in either direction
            if (c1, c2) not in existing_edges and (c2, c1) not in existing_edges:
                candidate_pairs.append((c1, c2))

print(f"Candidate pairs to check: {len(candidate_pairs)}")
print(f"At 10 pairs per API call: ~{len(candidate_pairs) // 10 + 1} API calls")

## The Claude edge-proposal prompt

This is the core of the system. We send Claude a batch of concept pairs and ask it to classify each one.
The prompt is carefully designed to produce structured JSON output.

In [ ]:
SYSTEM_PROMPT = """You are a calculus education expert helping build a knowledge graph of mathematical concepts.

For each pair of concepts, classify their relationship as ONE of:
- "prerequisite": Concept A must be understood before Concept B (A → B). Order matters!
- "uses": Concept B uses Concept A as a tool, but A isn't strictly needed to understand B
- "related_to": The concepts are analogous, inverse operations, or commonly confused
- "none": No meaningful direct relationship

IMPORTANT GUIDELINES:
- Be conservative: only mark "prerequisite" if a student truly CANNOT understand B without A
- "uses" means B applies A in practice but you could define B without knowing A
- "related_to" means thinking about one helps understand the other (analogies, inverses, contrasts)
- "none" is fine — not every pair needs a connection. Most pairs should be "none".
- For prerequisite and uses, specify the direction: which concept comes first?

Respond with ONLY a JSON array (no other text or markdown), where each element has:
{
  "concept_a": "...",
  "concept_b": "...",
  "relationship": "prerequisite" | "uses" | "related_to" | "none",
  "direction": "a_to_b" | "b_to_a" | "bidirectional" | null,
  "reason": "One sentence explaining why"
}"""


def build_batch_prompt(pairs):
    """Build a user prompt for a batch of concept pairs."""
    lines = ["Classify the relationship between each pair of calculus concepts:\n"]
    for i, (a, b) in enumerate(pairs, 1):
        desc_a = concepts[a].get("description", "")
        desc_b = concepts[b].get("description", "")
        lines.append(f"{i}. Concept A: \"{a}\" — {desc_a}")
        lines.append(f"   Concept B: \"{b}\" — {desc_b}")
        lines.append("")
    return "\n".join(lines)


def call_claude(pairs, max_retries=3):
    """Send a batch of pairs to Claude and parse the JSON response."""
    prompt = build_batch_prompt(pairs)

    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=2000,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}],
            )

            text = response.content[0].text.strip()
            # Strip markdown fences if present
            text = text.replace("```json", "").replace("```", "").strip()
            results = json.loads(text)
            return results

        except json.JSONDecodeError as e:
            print(f"  JSON parse error (attempt {attempt+1}): {e}")
            if attempt == max_retries - 1:
                print(f"  Raw response: {text[:500]}")
                return []
        except Exception as e:
            print(f"  API error (attempt {attempt+1}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return []

    return []


print("Prompt functions ready.")
print(f"\nExample prompt for first 3 pairs:\n")
print(build_batch_prompt(candidate_pairs[:3]))

## Run the edge discovery

This sends batches of 10 pairs to Claude. Each API call takes ~2-5 seconds.
Progress is saved after each batch so you can interrupt and resume.

**Cost estimate:** ~$0.50-1.00 total for ~100 pairs using Sonnet.

In [ ]:
BATCH_SIZE = 10
RESULTS_FILE = "edge_proposals.json"

# Load any existing progress
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        all_proposals = json.load(f)
    print(f"Loaded {len(all_proposals)} existing proposals")
else:
    all_proposals = []

# Figure out which pairs we've already processed
already_done = set()
for p in all_proposals:
    already_done.add((p["concept_a"], p["concept_b"]))

remaining = [(a, b) for a, b in candidate_pairs if (a, b) not in already_done]
print(f"Remaining pairs to process: {len(remaining)}")

# Process in batches
batches = [remaining[i:i+BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]

for batch_idx, batch in enumerate(batches):
    print(f"Batch {batch_idx+1}/{len(batches)} ({len(batch)} pairs)...", end=" ")

    results = call_claude(batch)

    # Count relationships found
    found = sum(1 for r in results if r.get("relationship") != "none")
    print(f"→ {found} relationships found")

    all_proposals.extend(results)

    # Save progress after each batch
    with open(RESULTS_FILE, "w") as f:
        json.dump(all_proposals, f, indent=2)

    # Rate limit: ~1 request per second is safe
    time.sleep(1)

print(f"\nDone. Total proposals: {len(all_proposals)}")

## Review the proposals

Claude's proposals need human review before being added to the graph.
This cell shows all non-"none" proposals for you to accept or reject.

In [ ]:
# Load proposals
with open(RESULTS_FILE, "r") as f:
    all_proposals = json.load(f)

# Filter to only meaningful relationships
meaningful = [p for p in all_proposals if p.get("relationship") not in ("none", None)]

print(f"Total proposals: {len(all_proposals)}")
print(f"Meaningful relationships found: {len(meaningful)}")
print()

# Group by relationship type
from collections import defaultdict
by_type = defaultdict(list)
for p in meaningful:
    by_type[p["relationship"]].append(p)

for rtype, proposals in sorted(by_type.items()):
    print(f"\n{'='*60}")
    print(f" {rtype.upper()} ({len(proposals)} proposed)")
    print(f"{'='*60}")
    for p in proposals:
        direction = p.get("direction", "")
        a, b = p["concept_a"], p["concept_b"]
        if direction == "b_to_a":
            a, b = b, a
        arrow = "→" if rtype in ("prerequisite", "uses") else "↔"
        print(f"  {a} {arrow} {b}")
        print(f"    Reason: {p.get('reason', 'N/A')}")

## Accept proposals and update the graph

After reviewing above, list the indices of proposals you want to REJECT.
Everything else gets added to the graph.

In [ ]:
# Print meaningful proposals with index numbers for easy rejection
print("REVIEW EACH PROPOSAL — note the indices of any you want to REJECT:\n")
for i, p in enumerate(meaningful):
    direction = p.get("direction", "")
    a, b = p["concept_a"], p["concept_b"]
    if direction == "b_to_a":
        a, b = b, a
    rtype = p["relationship"]
    arrow = "→" if rtype in ("prerequisite", "uses") else "↔"
    print(f"  [{i:3d}] {rtype:13s}  {a} {arrow} {b}")
    print(f"        Reason: {p.get('reason', '')}")

In [ ]:
# =============================================
# PUT THE INDICES YOU WANT TO REJECT HERE
# e.g., REJECT = {2, 5, 11, 14}
# Leave empty to accept all: REJECT = set()
# =============================================
REJECT = set()  # <-- Edit this!

accepted = [p for i, p in enumerate(meaningful) if i not in REJECT]
print(f"Accepting {len(accepted)} / {len(meaningful)} proposals\n")

# Add accepted proposals to the graph
added = 0
for p in accepted:
    direction = p.get("direction", "a_to_b")
    a, b = p["concept_a"], p["concept_b"]
    rtype = p["relationship"]

    if direction == "b_to_a":
        a, b = b, a

    # Skip if edge already exists
    if G.has_edge(a, b):
        continue

    # Skip if nodes don't exist (shouldn't happen but safe)
    if a not in G or b not in G:
        print(f"  Skipping (missing node): {a} → {b}")
        continue

    G.add_edge(a, b, edge_type=rtype, reason=p.get("reason", ""), source="claude_proposed")
    added += 1

    # For related_to, also add reverse edge
    if rtype == "related_to" and not G.has_edge(b, a):
        G.add_edge(b, a, edge_type=rtype, reason=p.get("reason", ""), source="claude_proposed")
        added += 1

print(f"Added {added} new edges to the graph")
print(f"Graph now: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Verify DAG property still holds for prerequisites
prereq_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'prerequisite']
prereq_G = nx.DiGraph(prereq_edges)
if nx.is_directed_acyclic_graph(prereq_G):
    print("Prerequisite subgraph is still a valid DAG.")
else:
    cycles = list(nx.simple_cycles(prereq_G))
    print(f"WARNING: {len(cycles)} cycle(s) introduced! Review and fix:")
    for c in cycles[:5]:
        print(f"  {' → '.join(c)}")

## Save the enriched graph

In [ ]:
import pandas as pd

# Save updated JSON
data = nx.node_link_data(G, edges="edges")
with open("calculus_kg_enriched.json", "w") as f:
    json.dump(data, f, indent=2)

# Save CSVs for Cosmograph
edges_df = pd.DataFrame([
    {"source": u, "target": v, "edge_type": d.get("edge_type", ""),
     "reason": d.get("reason", ""), "discovered_by": d.get("source", "manual")}
    for u, v, d in G.edges(data=True)
])
edges_df.to_csv("calculus_edges_enriched.csv", index=False)

nodes_df = pd.DataFrame([
    {"id": n, "type": d.get("type", ""), "description": d.get("description", ""),
     "hint": d.get("hint", ""), "in_degree": G.in_degree(n), "out_degree": G.out_degree(n)}
    for n, d in G.nodes(data=True)
])
nodes_df.to_csv("calculus_nodes_enriched.csv", index=False)

print(f"Saved:")
print(f"  calculus_kg_enriched.json ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")
print(f"  calculus_edges_enriched.csv ({len(edges_df)} edges)")
print(f"  calculus_nodes_enriched.csv ({len(nodes_df)} nodes)")

# Show breakdown of edge sources
source_counts = edges_df["discovered_by"].fillna("manual").value_counts()
print(f"\nEdge sources:")
for src, count in source_counts.items():
    print(f"  {src}: {count}")

## Compare: before and after

See what Claude discovered that we missed.

In [ ]:
# Show only the Claude-proposed edges
claude_edges = [(u, v, d) for u, v, d in G.edges(data=True) if d.get("source") == "claude_proposed"]
print(f"Claude discovered {len(claude_edges)} new edges:\n")

for u, v, d in sorted(claude_edges, key=lambda x: x[2].get("edge_type", "")):
    etype = d.get("edge_type", "")
    arrow = "→" if etype in ("prerequisite", "uses") else "↔"
    print(f"  [{etype:13s}] {u} {arrow} {v}")
    print(f"                 {d.get('reason', '')}")